# Pairwise CHRR distribution comparisons

Eight comparisons of saved chain-1 samples, including separate L3–L4f and L3–L4m
comparisons. The 93 fixed exclusions precede eligibility filtering: the sampled
flux range must be at least 0.001 in both models, in the source flux units.
KS D and the absolute median difference divided by the pooled sampled range
receive descending minimum ranks; their sum determines the original pandas
ascending sort order. P and BH-adjusted q values are descriptive and do not
select reactions. Relative paths resolve from the repository root.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

INPUT_DIR = Path(".")
OUTPUT_DIR = Path("results")
PCA_DIR = INPUT_DIR / "CHRR_sampling" / "pca"
RANGE_MIN = 0.001
PAIRS = [
    ("egg", "L1"), ("L1", "L2"), ("L2", "L3"), ("L3", "L4f"),
    ("L3", "L4m"), ("L4f", "Af"), ("L4m", "Am"), ("Af", "Am"),
]

# Fixed reaction exclusions recorded for the sampling analysis.
EXCLUDED_REACTIONS = """
MAR00023 MAR01506 MAR01507 MAR02960 MAR02993 MAR03031 MAR03036 MAR03041
MAR03131 MAR06630 MAR06631 MAR06632 MAR06633 MAR06634 MAR06635 MAR06636
MAR06637 MAR06638 MAR06639 MAR06641 MAR06642 MAR06646 MAR06647 MAR06648
MAR06649 MAR06651 MAR06652 MAR06653 MAR06654 MAR06655 MAR06656 MAR06657
MAR06658 MAR06659 MAR06660 MAR06662 MAR06663 MAR06664 MAR06665 MAR06666
MAR06667 MAR06668 MAR06669 MAR06670 MAR06671 MAR06672 MAR06673 MAR06674
MAR06675 MAR06676 MAR06677 MAR06678 MAR06680 MAR06682 MAR06685 MAR06686
MAR06687 MAR06690 MAR06691 MAR06692 MAR06693 MAR06694 MAR06695 MAR06697
MAR06699 MAR06700 MAR06701 MAR06702 MAR06703 MAR06704 MAR06705 MAR08697
MAR08698 MAR08699 MAR08702 MAR08703 MAR08704 MAR08706 MAR08709 MAR08710
MAR08711 MAR08712 MAR08716 MAR11894 MAR11895 MAR11896 MAR11897 MAR20001
MAR20003 MAR20005 MAR20006 MAR20007 MAR20008
""".split()


In [ ]:
STAGES = list(dict.fromkeys(stage for pair in PAIRS for stage in pair))
samples = {
    stage: pd.read_csv(PCA_DIR / f"{stage}_c_samples.csv")
             .drop(columns=EXCLUDED_REACTIONS, errors="ignore")
    for stage in STAGES
}


In [ ]:
# Annotations are optional and do not determine reaction eligibility.
annotations = pd.DataFrame(index=pd.Index([], name="rxnID"))
annotation_path = PCA_DIR / "reaction_evidence_table.csv"
if annotation_path.exists():
    annotations = pd.read_csv(annotation_path).drop_duplicates("rxnID").set_index("rxnID")
    columns = [c for c in ["probable_name", "subSystem", "eccodes", "equation"] if c in annotations]
    annotations = annotations[columns].rename(columns={"probable_name": "reaction_name"})


In [ ]:
# Original BH calculation, retained for descriptive reporting only.
def bh_adjust(pvals):
    ranks = pvals.rank(method="first")
    n = len(pvals)
    q = pvals * n / ranks
    running_min = 1.0
    q_adj = pd.Series(index=pvals.index, dtype=float)
    for idx in pvals.sort_values(ascending=False).index:
        running_min = min(running_min, q[idx])
        q_adj[idx] = running_min
    return q_adj.clip(upper=1.0)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ranked_tables = []
for order, (a, b) in enumerate(PAIRS, 1):
    pair = f"{a}_vs_{b}"
    x, y = samples[a], samples[b]
    rows = []
    for rid in sorted(set(x) & set(y)):
        va, vb = x[rid], y[rid]
        lo_a, hi_a, med_a = va.min(), va.max(), va.median()
        lo_b, hi_b, med_b = vb.min(), vb.max(), vb.median()
        if not (hi_a - lo_a >= RANGE_MIN and hi_b - lo_b >= RANGE_MIN):
            continue
        ks = ks_2samp(va.values, vb.values)
        delta = med_b - med_a
        pooled_range = max(hi_a, hi_b) - min(lo_a, lo_b)
        rows.append(dict(
            pair=pair, pair_order=order, model_A=a, model_B=b, rxnID=rid,
            CHRR_min_A=lo_a, CHRR_max_A=hi_a, median_A=med_a, range_A=hi_a-lo_a,
            CHRR_min_B=lo_b, CHRR_max_B=hi_b, median_B=med_b, range_B=hi_b-lo_b,
            n_samples_A=len(va), n_samples_B=len(vb),
            delta_median=delta, absolute_delta_median=abs(delta),
            pooled_range=pooled_range, relative_delta=abs(delta)/pooled_range,
            D=ks.statistic, pvalue=ks.pvalue,
        ))
    result = pd.DataFrame(rows)
    result["qvalue"] = bh_adjust(result["pvalue"])
    result["rank_D"] = result["D"].rank(ascending=False, method="min").astype(int)
    result["rank_relative_delta"] = result["relative_delta"].rank(ascending=False, method="min").astype(int)
    result["combined_score"] = result["rank_D"] + result["rank_relative_delta"]
    result = result.sort_values("combined_score").reset_index(drop=True)
    result["saved_table_position"] = np.arange(1, len(result) + 1)
    result["combined_score_rank_min"] = result["combined_score"].rank(method="min").astype(int)
    result["higher_median_in"] = np.where(
        result["delta_median"] > 0, b, np.where(result["delta_median"] < 0, a, "equal")
    )
    result = result.join(annotations, on="rxnID")
    result.to_csv(OUTPUT_DIR / f"pairwise_ks_{a}_c_vs_{b}_c.csv", index=False, float_format="%.17g")
    ranked_tables.append(result)


In [ ]:
pd.concat(ranked_tables, ignore_index=True).to_csv(
    OUTPUT_DIR / "sampling_pairwise_comparisons.tsv", sep="\t", index=False, float_format="%.17g"
)
